# 04. Compare test on synthetic

This notebook runs the full cross-validation benchmark on the synthetic tumor data.
It implements the exact methodology from the Apollo paper: Feature Selection (Top 20) + SVM.

In [1]:
import sys
import os
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
import joblib

# Add project root to path and fix working directory
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

os.chdir(project_root)

from src.data.loader import GliomaDataLoader
from src.models.baselines import ApolloPipeline
from src.utils.config_manager import ConfigManager
from src.notebooks.experiment_utils import setup_experiment

c:\Users\Andrei\anaconda3\envs\glioma\Lib\site-packages\h5py\__init__.py:36: UserWarning: h5py is running against HDF5 1.14.6 when it was built against 1.14.5, this may cause problems
  _warn(("h5py is running against HDF5 {0} when it was built against {1}, "


In [2]:
# Setup experiment from config
config_name = 'exp_105'
config, output_dir = setup_experiment(config_name)

# Load paths from config
DATA_PATH = config['paths']['processed_file']
SPLIT_FILE = config['paths']['splits_file']

print(f"✓ Config loaded: {config_name}")
print(f"Data path: {DATA_PATH}")
print(f"Splits file: {SPLIT_FILE}")

if not os.path.exists(SPLIT_FILE):
    print(f"❌ ERROR: Splits file not found at {SPLIT_FILE}")

print("Initializing Loader from config...")
loader = GliomaDataLoader.from_config(
    h5_data_path=DATA_PATH,
    config=config,
    memory_mapped=False
)

print(f"✓ Loader initialized. Total spectra: {loader.X.shape[0]}")


EXPERIMENT SETUP

Working directory: c:\Users\Andrei\incdsb\New folder\glioma-synthetic-data

 Loaded configuration: exp_105
   Description: TSAR Regimen: Fingerprint crop (700-1800), patient cap (500), and dynamic Raman augmentation

 Output directory: C:\Users\Andrei\incdsb\New folder\glioma-synthetic-data\results\results
  Experiment already registered (re-running)

--------------------------------------------------------------------------------
Configuration Summary:
--------------------------------------------------------------------------------
  Preprocessing:
    - AirPLS lambda: 100
    - Signal length: 1738
    - Crop region: [700, 1800]

  CV Strategy:
    - Split source: data/splits/strict_splits.json
    - Use group-aware CV: True
    - Hold out test set: True

  Models:
    - RF n_estimators: 100
    - SVM C: 1.0
    - CVAE epochs: 70


✓ Config loaded: exp_105
Data path: results/exp_102/data/glioma_dataset.h5
Splits file: data/splits/strict_splits.json
Initializing Load

In [5]:
import time
import os
import gc
import joblib
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
from src.models.cnn import build_1d_cnn 
from src.models.baselines import ApolloPipeline

# Load config paths and hyperparams
SYNTHETIC_IN_DIR = config['paths']['synthetic_dir']
MODEL_DIR = config['paths']['models_tstr_dir']
os.makedirs(MODEL_DIR, exist_ok=True)

# Extract model configs
apollo_cfg = config['models']['apollo']
rf_cfg = config['models']['random_forest']
xgb_cfg = config['models']['xgboost']
cnn_cfg = config['models']['cnn']
tstr_cfg = config.get('tstr', {})

# Check if binary classification
num_classes = config['models']['cnn']['num_classes']
if num_classes == 2:
    # Adjust XGBoost for binary classification
    xgb_binary_cfg = xgb_cfg.copy()
    xgb_binary_cfg['objective'] = 'binary:logistic'
    xgb_binary_cfg['num_classes'] = 1  # Binary uses num_class=1 (not 2)
else:
    xgb_binary_cfg = xgb_cfg

# Optional: sleep before training (for remote scheduling)
sleep_secs = tstr_cfg.get('sleep_before_training', 0)
if sleep_secs > 0:
    print(f"⏳ Sleeping for {sleep_secs} seconds before training...")
    time.sleep(sleep_secs)

tstr_benchmark_results = {
    'Apollo_TSTR': [],
    'RF_Standard_TSTR': [],
    'XGBoost_TSTR': [],
    '1D_CNN_TSTR': []
}

print("🚀 Starting TSTR (Train Synthetic, Test Real) Benchmark...")
print(f"Synthetic data dir: {SYNTHETIC_IN_DIR}")
print(f"Model output dir: {MODEL_DIR}")

for fold_data in loader:
    fold_name = fold_data['fold_name']
    print(f"\n{'='*60}")
    print(f"🔄 Processing {fold_name}")
    print(f"{'='*60}")
    
    # 1. LOAD SYNTHETIC TRAINING DATA
    print("Loading balanced synthetic data from CVAE...")
    X_train = np.load(f"{SYNTHETIC_IN_DIR}/{fold_name}_X_synth.npy").astype(np.float32)
    y_train = np.load(f"{SYNTHETIC_IN_DIR}/{fold_name}_y_synth.npy")
    
    # 2. Squeeze the CVAE channel dimension if necessary for ML models
    if len(X_train.shape) == 3:
        X_train = np.squeeze(X_train, axis=-1)
    
    print(f"   Synthetic data shape: {X_train.shape}, Labels: {y_train.shape}")
    
    # 3. UNPACK REAL VALIDATION DATA (Test Real)
    X_val = fold_data['val'][0].astype(np.float32)
    y_val = fold_data['val'][1]
    
    # Fix 1-indexed validation labels to 0-indexed
    if np.min(y_val) == 1:
        y_val = y_val - 1
    
    # Apply binary conversion to validation labels if needed (loader gives 6-class labels)
    if num_classes == 2:
        y_val = np.where(y_val < 3, 0, 1)
        print(f"  ✓ Converted validation labels to binary: (0, 1, 2) → 0, (3, 4, 5) → 1")
        
    # --- MODEL 1: Apollo Baseline ---
    print(f"Training Apollo (n_features={apollo_cfg['n_features']})...")
    apollo_pipeline = ApolloPipeline(
        n_features=apollo_cfg['n_features'],
        random_state=apollo_cfg['random_state']
    )
    apollo_pipeline.fit(X_train, y_train)
    y_pred_apollo = apollo_pipeline.predict(X_val)
    tstr_benchmark_results['Apollo_TSTR'].append(accuracy_score(y_val, y_pred_apollo))
    joblib.dump(apollo_pipeline, f"{MODEL_DIR}/apollo_{fold_name}.joblib")
    del y_pred_apollo
    gc.collect()
    
    # --- MODEL 2: Random Forest ---
    print(f"Training RF (n_est={rf_cfg['n_estimators']})...")
    rf = RandomForestClassifier(
        n_estimators=rf_cfg['n_estimators'],
        class_weight=rf_cfg['class_weight'],
        random_state=rf_cfg['random_state'],
        n_jobs=rf_cfg['n_jobs']
    )
    rf.fit(X_train, y_train)
    y_pred_rf = rf.predict(X_val)
    tstr_benchmark_results['RF_Standard_TSTR'].append(accuracy_score(y_val, y_pred_rf))
    joblib.dump(rf, f"{MODEL_DIR}/rf_{fold_name}.joblib")
    del rf, y_pred_rf
    gc.collect()
    
    # --- MODEL 3: XGBoost ---
    print(f"Training XGBoost...")
    xgb = XGBClassifier(
        objective=xgb_binary_cfg['objective'],
        num_class=xgb_binary_cfg.get('num_class', None),  # Use None if key doesn't exist
        random_state=xgb_binary_cfg['random_state'],
        n_jobs=xgb_binary_cfg['n_jobs'],
        tree_method=xgb_binary_cfg['tree_method']
    )
    xgb.fit(X_train, y_train)
    y_pred_xgb = xgb.predict(X_val)
    tstr_benchmark_results['XGBoost_TSTR'].append(accuracy_score(y_val, y_pred_xgb))
    xgb.save_model(f"{MODEL_DIR}/xgb_{fold_name}.json")
    del xgb, y_pred_xgb
    gc.collect()
    
    # --- MODEL 4: 1D-CNN ---
    print(f"Training 1D-CNN (epochs={cnn_cfg['epochs']})...")
    tf.keras.backend.clear_session()
    
    # Reshape for CNN - use ACTUAL feature dimension from data (after cropping)
    num_features = X_train.shape[1]
    X_train_cnn = X_train.reshape(-1, num_features, 1)
    X_val_cnn = X_val.reshape(-1, num_features, 1)
    y_train_cnn = tf.keras.utils.to_categorical(y_train, num_classes=cnn_cfg['num_classes'])
    y_val_cnn = tf.keras.utils.to_categorical(y_val, num_classes=cnn_cfg['num_classes'])
    
    cnn_model = build_1d_cnn(
        input_shape=(num_features, 1),
        num_classes=cnn_cfg['num_classes']
    )
    cnn_model.compile(
        optimizer=cnn_cfg['optimizer'],
        loss=cnn_cfg['loss'],
        metrics=['accuracy']
    )
    
    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=cnn_cfg['early_stopping_patience'],
        restore_best_weights=True
    )
    
    cnn_model.fit(X_train_cnn, y_train_cnn, 
                  validation_data=(X_val_cnn, y_val_cnn),
                  epochs=cnn_cfg['epochs'],
                  batch_size=cnn_cfg['batch_size'],
                  callbacks=[early_stop],
                  verbose=0)
                  
    y_pred_cnn_probs = cnn_model.predict(X_val_cnn, verbose=0)
    y_pred_cnn = np.argmax(y_pred_cnn_probs, axis=1)
    tstr_benchmark_results['1D_CNN_TSTR'].append(accuracy_score(y_val, y_pred_cnn))
    cnn_model.save(f"{MODEL_DIR}/cnn_{fold_name}.keras")
    del cnn_model, X_train_cnn, X_val_cnn, y_train_cnn, y_val_cnn, y_pred_cnn_probs, y_pred_cnn
    tf.keras.backend.clear_session()
    gc.collect()

# --- Print Final Summary ---
print("\n🎉 === TSTR BENCHMARK COMPLETE ===")
for model_name, accuracies in tstr_benchmark_results.items():
    mean_acc = np.mean(accuracies)
    std_acc = np.std(accuracies)
    print(f"{model_name:>20}: {mean_acc:.4f} ± {std_acc:.4f}")

🚀 Starting TSTR (Train Synthetic, Test Real) Benchmark...
Synthetic data dir: results/exp_105/synthetic
Model output dir: results/exp_105/models/tstr

 Using CV splits from: c:\Users\Andrei\incdsb\New folder\glioma-synthetic-data\data\splits\strict_splits.json
    Found 'cv_folds' section with 5 folds
   ! Separate held-out test set exists: 6 samples
   !  These samples will NOT be used in CV training/validation
   Fold 0: 160084 train samples, 45936 val samples

🔄 Processing fold_0
Loading balanced synthetic data from CVAE...
   Synthetic data shape: (40000, 571), Labels: (40000,)
  ✓ Converted validation labels to binary: (0, 1, 2) → 0, (3, 4, 5) → 1
Training Apollo (n_features=20)...
  [Apollo] RF Config: n_estimators=300, max_depth=10, n_features_selected=20
  [Apollo] SVM Config: C=1.0, kernel=rbf
  [Apollo] Fitting RF Selector on (40000, 571) features...
  [Apollo] Selected 20 features.
  [Apollo] Fitting SVM Classifier...
Training RF (n_est=100)...
Training XGBoost...
Training 1

In [5]:
# === DIAGNOSTICS: Check Synthetic Data Properties ===
print("\n" + "="*60)
print("SYNTHETIC DATA DIAGNOSTICS")
print("="*60)

# Load one fold of synthetic data to inspect
synthetic_fold = "fold_0"
X_synth = np.load(f"{SYNTHETIC_IN_DIR}/{synthetic_fold}_X_synth.npy")
y_synth = np.load(f"{SYNTHETIC_IN_DIR}/{synthetic_fold}_y_synth.npy")

# Also load corresponding real validation data for comparison
for fold_data in loader:
    if fold_data['fold_name'] == synthetic_fold:
        X_real_val = fold_data['val'][0].astype(np.float32)
        y_real_val = fold_data['val'][1]
        break

print(f"\nSynthetic Data ({synthetic_fold}):")
print(f"  Shape: {X_synth.shape}")
print(f"  Dtype: {X_synth.dtype}")
print(f"  Value range: [{np.min(X_synth):.4f}, {np.max(X_synth):.4f}]")
print(f"  Labels: {y_synth.shape}, unique values: {np.unique(y_synth)}")
print(f"  Label distribution: {np.bincount(y_synth)}")

print(f"\nReal Validation Data ({synthetic_fold}):")
print(f"  Shape: {X_real_val.shape}")
print(f"  Dtype: {X_real_val.dtype}")
print(f"  Value range: [{np.min(X_real_val):.4f}, {np.max(X_real_val):.4f}]")
if np.min(y_real_val) == 1:
    y_real_val_0idx = y_real_val - 1
else:
    y_real_val_0idx = y_real_val
print(f"  Labels (0-indexed): unique values: {np.unique(y_real_val_0idx)}")
print(f"  Label distribution: {np.bincount(y_real_val_0idx)}")

print(f"\n✓ Feature dimension match: {X_synth.shape[1]} == {X_real_val.shape[1]}")
print(f"✓ Synthetic is class-balanced: {len(np.unique(y_synth))} classes, each with {np.bincount(y_synth)[0]} samples")
print(f"✓ Both cropped to fingerprint region: [{X_synth.shape[1]} features]")

print("="*60)


SYNTHETIC DATA DIAGNOSTICS

 Using CV splits from: c:\Users\Andrei\incdsb\New folder\glioma-synthetic-data\data\splits\strict_splits.json
    Found 'cv_folds' section with 5 folds
   ! Separate held-out test set exists: 6 samples
   !  These samples will NOT be used in CV training/validation
   Fold 0: 160084 train samples, 45936 val samples

Synthetic Data (fold_0):
  Shape: (120000, 571, 1)
  Dtype: float32
  Value range: [0.0184, 0.7472]
  Labels: (120000,), unique values: [0 1 2 3 4 5]
  Label distribution: [20000 20000 20000 20000 20000 20000]

Real Validation Data (fold_0):
  Shape: (45936, 571)
  Dtype: float32
  Value range: [0.0000, 1.0000]
  Labels (0-indexed): unique values: [0 1 2 3 4 5]
  Label distribution: [ 2324 14645  3261  9787 12614  3305]

✓ Feature dimension match: 571 == 571
✓ Synthetic is class-balanced: 6 classes, each with 20000 samples
✓ Both cropped to fingerprint region: [571 features]


In [4]:
%reset -f